In [ ]:
"""
VCO frequency-vs-code characterization: interpolation-based version.

Same input data and the same overall figure / lookup-table outputs as the
polynomial-regression script, but the forward (Code -> Freq) and inverse
(Freq -> Code) models are now built with monotonic-preserving piecewise
cubic Hermite interpolation (scipy.interpolate.PchipInterpolator) instead
of a fitted 2nd-order polynomial. PCHIP passes exactly through every
measured point and, unlike a plain cubic spline, cannot introduce
non-physical overshoot / non-monotonic wiggles between points -- important
here since the code is used as a monotonic calibration lookup.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

# =============================================================================
# 1. LOAD AND PARSE THE CSV DATA
# =============================================================================
csv_file = 'TRANSIENT_COMP_ALL_NEWVAR_PrimeSim_default_VCO_comp_bck2_Measurements_history_1_20260615_22_50_31.99.csv'
df_raw = pd.read_csv(csv_file)

def parse_freq(val):
    if pd.isna(val): return np.nan
    val = str(val).strip()
    
    try:
        if val.endswith('G'): return float(val[:-1])
        if val.endswith('M'): return float(val[:-1]) / 1e3
        return float(val)
    except ValueError:
        # If PrimeSim outputs 'running', 'failed', or any non-number, return NaN.
        # This allows dropna() to cleanly remove incomplete simulation points later.
        return np.nan

def parse_code(val):
    if pd.isna(val): return np.nan
    val = str(val).strip()
    
    try:
        # Handle explicit millivolt suffix (e.g., '15m' -> 15.0)
        if val.endswith('m'): 
            return float(val[:-1])
            
        v = float(val)
        
        # PrimeSim drops the 'm' for values >= 100mV and switches to Volts.
        # Any small float (e.g., 0.1, 0.11, 0.127) needs to be multiplied 
        # by 1000 to convert to mV/integer code.
        if 0 < v < 2.0:
            return v * 1000.0
            
        return v
    except ValueError:
        return np.nan

def parse_process(val):
    if 'Typ' in str(val): return 'Typical'
    if 'bcQ' in str(val): return 'Bcq (Best)'
    if 'wcQ' in str(val): return 'Wcq (Worst)'
    return 'Unknown'

def to_inverted_binary(code):
    # Standard 7-bit binary representation of the code
    return f"{int(code):07b}"

# Build the cleaned DataFrame
df = pd.DataFrame()
df['Process'] = df_raw['Corner'].apply(parse_process)
df['Temp'] = df_raw['Corner:Variable:temp'].astype(float)
df['Code'] = df_raw['Corner:Variable:Vref'].apply(parse_code)
df['Freq_GHz'] = df_raw['AvgFreq:tran'].apply(parse_freq)

# Drop invalid rows (this removes the 'running' data points)
df = df.dropna(subset=['Freq_GHz', 'Code'])

# =============================================================================
# 2. PLOTTING SETUP & TABLE GENERATION
# =============================================================================
REF_FREQS = [6.2, 6.5, 6.8, 7.1, 7.4, 7.7, 8.0, 8.3, 8.6, 8.9]

fig, ax = plt.subplots(figsize=(12, 8))

COLORS = {'Typical': 'green', 'Bcq (Best)': 'blue', 'Wcq (Worst)': 'red'}
MARKERS = {-40.0: '^', 25.0: 'o', 125.0: 's'}
LINESTYLES = {-40.0: '--', 25.0: '-', 125.0: ':'}

# Extract unique temperatures directly from the parsed data
TARGET_TEMPS = sorted(df['Temp'].unique())

for temp in TARGET_TEMPS:
    print(f"\n{'='*80}\nTARGET CODES & ACTUAL FREQUENCIES FOR {int(temp)}°C\n{'='*80}")
    print(f"{'Process':<15} | {'Target (GHz)':<15} | {'Required Code':<15} | {'Binary (7-bit)':<15} | {'Actual AvgFreq (GHz)':<20}")
    print("-" * 80)

    for process in ['Typical', 'Bcq (Best)', 'Wcq (Worst)']:
        sub_df = df[(df['Process'] == process) & (df['Temp'] == temp)]
        if sub_df.empty:
            continue

        # Average out any duplicate codes and sort
        sub_grouped = sub_df.groupby('Code')['Freq_GHz'].mean().reset_index().sort_values('Code')
        x_vals, y_vals = sub_grouped['Code'].values, sub_grouped['Freq_GHz'].values

        # Forward interpolant (Code -> Freq): shape-preserving piecewise cubic
        # Hermite interpolation, passes exactly through every measured point
        interp_fwd = PchipInterpolator(x_vals, y_vals)
        x_smooth = np.linspace(min(x_vals), max(x_vals), 200)
        y_smooth = interp_fwd(x_smooth)

        # Safely get marker and linestyle just in case temp isn't exactly -40, 25, or 125
        marker = MARKERS.get(temp, 'o')
        linestyle = LINESTYLES.get(temp, '-')

        ax.plot(x_smooth, y_smooth, color=COLORS.get(process, 'black'), linestyle=linestyle,
                linewidth=2, alpha=0.85, label=f"{process} @ {int(temp)}°C")
        ax.plot(x_vals, y_vals, marker=marker, color=COLORS.get(process, 'black'),
                linestyle='none', markersize=6)

        # Inverse interpolant (Freq -> Code) for looking up target codes.
        # PCHIP requires strictly increasing, duplicate-free x-values, so any
        # repeated frequency values are first collapsed (averaging their
        # corresponding codes) before building the interpolant.
        inv_grouped = (pd.DataFrame({'Freq_GHz': y_vals, 'Code': x_vals})
                        .groupby('Freq_GHz')['Code'].mean()
                        .reset_index().sort_values('Freq_GHz'))
        y_sorted, x_sorted = inv_grouped['Freq_GHz'].values, inv_grouped['Code'].values
        interp_inv = PchipInterpolator(y_sorted, x_sorted)

        for f_target in REF_FREQS:
            # Interpolate the required code and bound it between [0, 127]
            code = int(np.round(np.clip(interp_inv(f_target), 0, 127)))
            actual_freq = float(interp_fwd(code))
            bin_str = to_inverted_binary(code)
            print(f"{process:<15} | {f_target:<15} | {code:<15} | {bin_str:<15} | {actual_freq:<20.4f}")

# --------------------------------------------------------
# Plot target reference curves and formatting
# --------------------------------------------------------
# Adding horizontal target frequency lines
for freq in REF_FREQS:
    ax.axhline(y=freq, color='black', linestyle=':', linewidth=0.8, alpha=0.4)
    # Added ha='right' so the text stays inside the plot space on the right edge
    ax.text(x=ax.get_xlim()[1], y=freq, s=f'{freq:.1f} GHz ', fontsize=8, va='bottom', ha='right')

ax.set_title('VCO Frequency vs Actual Digital Code', fontsize=14, fontweight='bold')
ax.set_xlabel('Digital Code (Decimal equivalent)', fontsize=12)
ax.set_ylabel('Frequency (GHz)', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.6)

# Changed legend location to 'best' (or 'upper left') and removed the external bounding box
ax.legend(loc='best', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:

# ============================================================
# CONFIGURATION
# ============================================================
CSV_FILE = "TRANSIENT_COMP_DELTAFREQ_NEWVAR_PrimeSim_default_VCO_comp_bck2_Measurements_history_1_20260616_22_30_45.31.csv"
PROCESS_ORDER = ["Typical", "Bcq (Best)", "Wcq (Worst)"]
UNIT_MAP = {'f': 1e-15, 'p': 1e-12, 'n': 1e-9, 'u': 1e-6, 'm': 1e-3, 'k': 1e3, 'M': 1e6, 'G': 1e9}

# Restored Plotting Configs
PLOT_TEMPS = [-40.0, 125.0]
TARGET_FREQS = np.array([6.2, 6.5, 6.8, 7.1, 7.4, 7.7, 8.0, 8.3, 8.6, 8.9])
PLOT_COLORS = {"Typical": "green", "Bcq (Best)": "blue", "Wcq (Worst)": "red"}
PLOT_MARKERS = {"Typical": "o", "Bcq (Best)": "d", "Wcq (Worst)": "^"}

# ============================================================
# DATA PARSING FUNCTIONS
# ============================================================
def parse_units(value):
    if pd.isna(value) or str(value).strip().lower() in ["", "error"]:
        return np.nan
    val_str = str(value).strip()
    match = re.search(r'([0-9\.-]+)([a-zA-Z]*)', val_str)
    if match:
        number, suffix = float(match.group(1)), match.group(2)
        return number * UNIT_MAP.get(suffix, 1.0)
    return np.nan

def get_process(corner_name):
    c = str(corner_name).lower()
    if "typ" in c: return "Typical"
    if "bcq" in c: return "Bcq (Best)"
    if "wcq" in c: return "Wcq (Worst)"
    return None

# ============================================================
# LOAD & PREPARE DATA
# ============================================================
df = pd.read_csv(CSV_FILE)
df["Freq_GHz"] = df["AvgFreq:tran"].apply(parse_units) / 1e9
df["Code"] = df["Corner:Variable:Vref"].apply(parse_units) * 1000
df["Temp"] = pd.to_numeric(df["Corner:Variable:temp"], errors="coerce")
df["Process"] = df["Corner"].apply(get_process)
df = df.dropna(subset=["Process"])

# ============================================================
# DELTA FREQUENCY ANALYSIS [ F(-40) - F(125) ]
# ============================================================
print("=" * 80)
print(f"{'Process':<15} | {'Code (mV)':<10} | {'F(-40) GHz':<12} | {'F(125) GHz':<12} | {'Delta Freq (MHz)':<15}")
print("=" * 80)

for process in PROCESS_ORDER:
    # Filter by process and temperature
    df_m40 = df[(df["Process"] == process) & (df["Temp"] == -40.0)].sort_values("Code")
    df_125 = df[(df["Process"] == process) & (df["Temp"] == 125.0)].sort_values("Code")
    
    # Merge to ensure we are comparing the exact same codes
    merged = pd.merge(df_m40, df_125, on="Code", suffixes=('_m40', '_125'))
    
    for _, row in merged.iterrows():
        code = row["Code"]
        f_m40 = row["Freq_GHz_m40"]
        f_125 = row["Freq_GHz_125"]
        delta_mhz = (f_m40 - f_125) * 1000  # Convert delta GHz to MHz
        
        print(f"{process:<15} | {code:<10.0f} | {f_m40:<12.4f} | {f_125:<12.4f} | {delta_mhz:<15.2f}")
    
    # Visual separator between process corners
    if not merged.empty:
        print("-" * 80)

# ============================================================
# PLOTTING (-40°C and 125°C)
# ============================================================
for temp in PLOT_TEMPS:
    fig, ax = plt.subplots(figsize=(10, 6))
    pts = np.arange(1, len(TARGET_FREQS) + 1)
    
    # Target Reference Line
    ax.plot(pts, TARGET_FREQS, '--o', linewidth=3, markersize=8, color='purple', label='Target')
    
    # Corners
    for process in PROCESS_ORDER:
        # Group to average any duplicate codes, then sort by frequency to align sequentially with points
        sub = df[(df["Process"] == process) & (df["Temp"] == temp)].groupby("Code")["Freq_GHz"].mean().reset_index().sort_values("Freq_GHz")
        if not sub.empty:
            # We assume the number of extracted codes matches the length of TARGET_FREQS (10 points)
            # If length mismatches, we cap it to the minimum length to avoid dimension errors
            plot_pts = pts[:len(sub)] 
            ax.plot(plot_pts, sub["Freq_GHz"], marker=PLOT_MARKERS.get(process, 'o'), 
                    color=PLOT_COLORS.get(process, 'black'), linewidth=2, markersize=8, label=process)
            
    ax.set_title(f'VCO Frequency vs Target Point ({int(temp)}°C)', fontweight='bold')
    ax.set_xlabel('Target Frequency Point (1 to 10)')
    ax.set_ylabel('Frequency (GHz)')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend()
    plt.tight_layout()
    plt.show()